In [ ]:
max_samples = 100

In [ ]:
input_file_dataset_filtered_full =      "dataset_filtered_full.xlsx"
input_file_dataset_filterd_dropped =    f"dataset_filtered_{max_samples}_dropped.xlsx"

input_file_queries_in_dataset =                         f"queries_{max_samples}_uniques_in_dataset.xlsx"
input_file_queries_similar_dataset_within_context =     f"queries_{max_samples}_uniques_similar_dataset_within_context.xlsx"
input_file_queries_similar_dataset_without_context =    f"queries_{max_samples}_uniques_similar_dataset_without_context.xlsx"

file_retrieved_contexts_ = f'retrieved_contexts_{max_samples}_'
file_conversation_ = f'conversation_{max_samples}_'
file_answer_ = f'answer_{max_samples}_'
file_response_ = f'response_{max_samples}_'

file_evaluations_ = f'evaluations_{max_samples}_'

DATASET_FILTERED_FULL_CONTEXT_EMBEDDINGS_FILE =   "dataset_filtered_full_context_embeddings.pt"    #File to save embeddings
DATASET_FILTERED_FULL_QUESTION_EMBEDDINGS_FILE =  "dataset_filtered_full_question_embeddings.pt"   #File to save embeddings
DATASET_FILTERED_FULL_QUESTION_INDEX_FILE =       "dataset_filtered_full_question_faiss_index.bin" #File to save index

DATASET_FILTERED_DROPPED_CONTEXT_EMBEDDINGS_FILE =   "dataset_filtered_dropped_context_embeddings.pt"    #File to save embeddings
DATASET_FILTERED_DROPPED_QUESTION_EMBEDDINGS_FILE =  "dataset_filtered_dropped_question_embeddings.pt"   #File to save embeddings
DATASET_FILTERED_DROPPED_QUESTION_INDEX_FILE =       "dataset_filtered_dropped_question_faiss_index.bin" #File to save index

In [ ]:
# version = 'in_dataset'
# version = 'out_dataset'
# version = 'similar_dataset_within_context'
version = 'similar_dataset_without_context'

if version == 'in_dataset':
    input_file_dataset = input_file_dataset_filtered_full
    input_file_queries = input_file_queries_in_dataset

    DATASET_CONTEXT_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_CONTEXT_EMBEDDINGS_FILE
    DATASET_QUESTION_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_QUESTION_EMBEDDINGS_FILE
    DATASET_QUESTION_INDEX_FILE = DATASET_FILTERED_FULL_QUESTION_INDEX_FILE
elif version == 'out_dataset':
    input_file_dataset = input_file_dataset_filterd_dropped
    input_file_queries = input_file_queries_in_dataset

    DATASET_CONTEXT_EMBEDDINGS_FILE = DATASET_FILTERED_DROPPED_CONTEXT_EMBEDDINGS_FILE
    DATASET_QUESTION_EMBEDDINGS_FILE = DATASET_FILTERED_DROPPED_QUESTION_EMBEDDINGS_FILE
    DATASET_QUESTION_INDEX_FILE = DATASET_FILTERED_DROPPED_QUESTION_INDEX_FILE
elif version == 'similar_dataset_within_context':
    input_file_dataset = input_file_dataset_filtered_full
    input_file_queries = input_file_queries_similar_dataset_within_context

    DATASET_CONTEXT_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_CONTEXT_EMBEDDINGS_FILE
    DATASET_QUESTION_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_QUESTION_EMBEDDINGS_FILE
    DATASET_QUESTION_INDEX_FILE = DATASET_FILTERED_FULL_QUESTION_INDEX_FILE
elif version == 'similar_dataset_without_context':
    input_file_dataset = input_file_dataset_filtered_full
    input_file_queries = input_file_queries_similar_dataset_without_context

    DATASET_CONTEXT_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_CONTEXT_EMBEDDINGS_FILE
    DATASET_QUESTION_EMBEDDINGS_FILE = DATASET_FILTERED_FULL_QUESTION_EMBEDDINGS_FILE
    DATASET_QUESTION_INDEX_FILE = DATASET_FILTERED_FULL_QUESTION_INDEX_FILE

middle_file_retrieved_contexts = file_retrieved_contexts_ + version + '.xlsx'
middle_file_conversation = file_conversation_ + version + '.xlsx'
middle_file_answer = file_answer_ + version + '.xlsx'
middle_file_response = file_response_ + version + '.xlsx'

output_file_evaluations = file_evaluations_ + version + '.xlsx'

summary_file_reordered_True = f"evaluation_summary_{max_samples}_True_" + version + ".xlsx"
summary_file_reordered_False = f"evaluation_summary_{max_samples}_False_" + version + ".xlsx"

# Import libraries and models

In [ ]:
import numpy as np
import pandas as pd
import ast

In [ ]:
import openai
import json
import os

# Load OpenAI API key
key_path = './config.json'
with open(key_path, "r") as file:
    config = json.load(file)
os.environ["OPENAI_API_KEY"] = config["openai_api_key"]

openai.api_key = config["openai_api_key"]

In [ ]:
# !pip install faiss-cpu
import faiss
import torch

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('distiluse-base-multilingual-cased-v1')

from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 34.8 MB/s eta 0:00:00


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# !pip install ragas
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, SemanticSimilarity
from ragas.llms import LangchainLLMWrapper

from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

from langchain_openai import ChatOpenAI
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", api_key=config["openai_api_key"]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

# Define functions

In [ ]:
def create_embeddings(sentences, EMBEDDINGS_FILE: str=None):
    """Creates sentence embeddings and saves them to a file."""
    embeddings = model.encode(sentences, convert_to_tensor=True)

    if EMBEDDINGS_FILE is not None:
        torch.save(embeddings, EMBEDDINGS_FILE) #save to file
    return embeddings

def load_embeddings(EMBEDDINGS_FILE):
    """Loads embeddings from a file or creates them if the file doesn't exist."""
    if os.path.exists(EMBEDDINGS_FILE):
        print("Loading embeddings from file...")
        embeddings = torch.load(EMBEDDINGS_FILE)
        return embeddings
    else:
        return None  # Indicate that embeddings need to be created

def create_faiss_index(embeddings, INDEX_FILE: str=None):
    """Creates a FAISS index and saves it to a file."""
    embeddings_np = embeddings.cpu().detach().numpy()
    dimension = embeddings_np.shape[1]
    index = faiss.IndexFlatL2(dimension)  # Or IndexIVFFlat for larger datasets
    index.add(embeddings_np)

    if INDEX_FILE is not None:
        faiss.write_index(index, INDEX_FILE) #save to file
    return index

def load_faiss_index(INDEX_FILE):
    """Loads a FAISS index from a file or creates it if the file doesn't exist."""
    if os.path.exists(INDEX_FILE):
        print("Loading FAISS index from file...")
        index = faiss.read_index(INDEX_FILE)
        return index
    else:
        return None  # Indicate that index needs to be created


def call_chatgpt(user_message: dict,
        sys_message={"role": "system", "content": "You are a helpful assistant that responds in Vietnamese."},
        model="gpt-4o-mini",
        max_tokens=1000,
        temperature=0):
    response = openai.chat.completions.create(
        model=model,  # Use the appropriate model
        messages=[sys_message, user_message],
        max_tokens=max_tokens,  # Further reduced to ensure brevity
        temperature=temperature  # Keep at 0 for precision
    )

    # Get the content of the response
    response_content = response.choices[0].message.content.strip()

    # Return the extracted answer directly
    return response_content

def generate_answer(context: str, question: str):
    sys_message = {"role": "system", "content": "You are a helpful assistant that responds in Vietnamese."}

    user_content = f"Context: {context}\nQuestion: {question}\nPlease only extract the exact continuous words from the context that answers the question. For example, context: Đến năm 1327, đây là thị trấn lớn thứ ba tại Warwickshire, question: Vào thế kỉ XIV, Birmingham trở thành thị trấn lớn thứ mấy tại Warwickshire?, answer: lớn thứ ba, not: thị trấn lớn thứ ba."
    user_message = {"role": "user", "content": user_content}

    chatgpt_response = call_chatgpt(user_message, sys_message)
    print("ChatGPT: ", chatgpt_response)

    return chatgpt_response


def format_retrieved_contexts(query: str, contexts: list[str]) -> dict:
    return {"user_query": query,
            "retrieved_contexts": contexts}

def combine_str_lst(str_lst: list[str], sep: str=' ') -> str:
    return sep.join(str_lst)

def format_conversation(query: str, contexts: list[str]) -> dict:
    combined_context = combine_str_lst(contexts)
    return {"role": "user",
            "content": f"Question: {query}\nContext: {combined_context}",
            "combined_context": f"{combined_context}"}

def format_response(query: str, contexts: list[str], answer: str) -> dict:
    return {"user_query": query,
            "retrieved_contexts": contexts,
            "response": answer}

def format_evaluation_input(user_query: str, retrieved_contexts: list[str], response: str, reference: str):
    return {"user_input": user_query,
            "retrieved_contexts": retrieved_contexts,
            "response": response,
            "reference": reference}


def get_answer_text_in_str(answer_str: str) -> str:
    start_pos = len("{'text': ['")
    end_pos = answer_str.find('answer_start')
    return answer_str[start_pos:end_pos-len("'], '")]

def get_answer_texts_in_df(df: pd.DataFrame) -> list[str]:
    answer_strs = df['answers'].tolist()
    return [get_answer_text_in_str(answer_str) for answer_str in answer_strs]

def get_strs_in_df(df: pd.DataFrame, attribute_str: str) -> list[str]:
    return df[attribute_str].tolist()

def get_str_lists_in_df(df: pd.DataFrame, attribute_str: str) -> list[str]:
    return df[attribute_str].apply(ast.literal_eval).tolist()


def save_lists_dict(filename, lists_dict: dict):
    """
    Saves multiple lists dictionary to an Excel file with column names.
    Format dict: {"column_name": list}
    """
    # Create a pandas DataFrame from the dictionary of lists
    df = pd.DataFrame(lists_dict)

    # Save the DataFrame to an Excel file
    df.to_excel(filename, index=False, engine='openpyxl')  # Explicitly use openpyxl

    print(f"Data saved to {filename}")

def save_dict_lst(filename:str, dict_lst: list[dict], column_names: list[str]=[]):
    # list of dictionaries to DataFrame
    if column_names == []:
        df = pd.DataFrame(dict_lst)
    else:
        df = pd.DataFrame(dict_lst, columns=column_names)

    # Save to Excel
    df.to_excel(filename, index=False)


def retrieve_top_k(query: str, contexts: list[str], index, k: int=5) -> list[str]:
    # Step 1: Retrieve initial top-k candidates using the bi-encoder
    query_embedding = create_embeddings(query)
    query_embedding_np = query_embedding.cpu().detach().numpy()

    # Step 1.5: Search the index
    distances, indices = index.search(np.array([query_embedding_np]), k)

    # Step 2: Prepare pairs for cross-encoder reranking (using a set)
    seen_contexts = set()  # Create an empty set to track seen contexts
    candidate_pairs = []
    for i in indices[0]:
        context = contexts[int(i)]
        if context not in seen_contexts:  # Check if context is already seen
            seen_contexts.add(context)  # Add context to the set if not seen
            candidate_pairs.append((query, context))

    # Step 3: Use cross-encoder to score and rerank the candidates
    scores = cross_encoder.predict(candidate_pairs)
    ranked_indices = np.argsort(scores)[::-1]  # Sort scores in descending order

    # Step 4: Return the reranked results
    top_k_contexts = [contexts[int(indices[0][i])] for i in ranked_indices]
    return top_k_contexts


def create_summary(df: pd.DataFrame, only_factual: bool, summary_file_reordered: str="evaluation_summary.xlsx"):
    """Calculates and exports a summary of evaluation metrics."""

    # Calculate means for numeric columns only
    column_means = df.select_dtypes(include=np.number).mean(skipna=True) #select only numeric columns

    # Calculate counts for all columns
    column_counts = df.count()

    # Create a DataFrame for means and counts, using only the numeric columns for means
    summary_df = pd.DataFrame({
        'Mean': column_means,
        'Count': column_counts
    })

    # Define the desired order of the metrics
    if only_factual:
        desired_order = ['factual_correctness', 'user_input', 'retrieved_contexts', 'response', 'reference']
    else:
        desired_order = ['context_recall', 'factual_correctness', 'faithfulness', 'semantic_similarity',
                         'user_input', 'retrieved_contexts', 'response', 'reference']

    # Reorder the rows in the summary DataFrame. Handle missing indices gracefully.
    summary_df_reordered = summary_df.reindex(desired_order)

    # Export the reordered summary to an Excel file
    summary_df_reordered.to_excel(summary_file_reordered, index=True)

    print(f"Reordered summary exported to {summary_file_reordered}")

# Load dataset

In [ ]:
dataset_df = pd.read_excel(input_file_dataset)

dataset_contexts = get_strs_in_df(dataset_df, 'context')
dataset_questions = get_strs_in_df(dataset_df, 'question')
dataset_answers = get_answer_texts_in_df(dataset_df)
print(dataset_contexts[:10])
print(dataset_questions[:10])
print(dataset_answers[:10])

['Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', 'Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng 

### Load embeddings and FAISS index

In [ ]:
# Load embeddings if they exist, otherwise create and save them
dataset_context_embeddings = load_embeddings(DATASET_CONTEXT_EMBEDDINGS_FILE)
if dataset_context_embeddings is None:
    dataset_context_embeddings = create_embeddings(dataset_contexts, DATASET_CONTEXT_EMBEDDINGS_FILE)
    print("Dataset context embeddings created and saved.")
else:
    print("Dataset context embeddings loaded from file.")

# Load embeddings if they exist, otherwise create and save them
dataset_question_embeddings = load_embeddings(DATASET_QUESTION_EMBEDDINGS_FILE)
if dataset_question_embeddings is None:
    dataset_question_embeddings = create_embeddings(dataset_questions, DATASET_QUESTION_EMBEDDINGS_FILE)
    print("Dataset question embeddings created and saved.")
else:
    print("Dataset question embeddings loaded from file.")

# Load index if it exists, otherwise create and save it
dataset_question_index = load_faiss_index(DATASET_QUESTION_INDEX_FILE)
if dataset_question_index is None:
    dataset_question_index = create_faiss_index(dataset_question_embeddings, DATASET_QUESTION_INDEX_FILE)
    print("Dataset question FAISS index created and saved.")
else:
    print("Dataset question FAISS index loaded from file.")

Loading embeddings from file...
Dataset context embeddings loaded from file.
Loading embeddings from file...
Dataset question embeddings loaded from file.
Loading FAISS index from file...
Dataset question FAISS index loaded from file.


C:\Users\USER\AppData\Local\Temp\ipykernel_2604\2960238826.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings = torch.load(EMBEDDINGS_FILE)


# Load queries

In [ ]:
# LOAD QUERIES IF UPDATED
queries_df = pd.read_excel(input_file_queries)

queries_contexts = get_strs_in_df(queries_df, 'context')
queries_questions = get_strs_in_df(queries_df, 'question')
queries_answers = get_strs_in_df(queries_df, 'answer')
print(queries_contexts[:10])
print(queries_questions[:10])
print(queries_answers[:10])

['Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', 'Năm 1954, ông được giao nhiệm vụ Trưởng phái đoàn Chính phủ dự Hội nghị Genève về Đông Dương. Những đóng góp của đoàn Việt Nam do ông đứng đầu là vô cùng quan trọng, tạo ra những đột phá đưa Hội nghị tới thành công. Trải qua 8 phiên họp toàn thể và 23 phiên họp rất căng thẳng và phức tạp, với tinh thần chủ động và cố gắng của phái đoàn Việt Nam, ngày 20/7/1954, bản Hiệp định đình chỉ chiến sự 

# Gen retrieved contexts

In [ ]:
retrieved_contexts = []
for queries_question in queries_questions:
    # if query is in dataset, look it up
    if queries_question in dataset_questions:
        query_index = dataset_questions.index(queries_question)
        retrieved_contexts.append(format_retrieved_contexts(queries_question, [dataset_contexts[query_index]]))
    # else, retrieve top k
    else:
        top_k_contexts = retrieve_top_k(queries_question, dataset_contexts, dataset_question_index)
        retrieved_contexts.append(format_retrieved_contexts(queries_question, top_k_contexts))

print(retrieved_contexts[:10])
save_dict_lst(middle_file_retrieved_contexts, retrieved_contexts)

[{'user_query': 'Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?', 'retrieved_contexts': ['Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', "Trong khi chính phủ Pháp đang do dự về yêu cầu của Cao ủy Đông Dương Georges D'Argenlieu về việc tăng quân và lập tức hành động quân sự chống lại người Việt, Valluy, người có chung quyết tâm với d'Argenlieu về việc giữ sự hiện diện của Pháp tại Đông Dương, đã quyết định rằng cần phải khiêu 

# Gen conversation

In [ ]:
retrieved_contexts_df = pd.read_excel(middle_file_retrieved_contexts)

retrieved_contexts_queries = get_strs_in_df(retrieved_contexts_df, 'user_query')
retrieved_contexts_contexts = get_str_lists_in_df(retrieved_contexts_df, 'retrieved_contexts')

In [ ]:
conversations = [format_conversation(query, contexts) for query, contexts in zip(retrieved_contexts_queries, retrieved_contexts_contexts)]

print(conversations[:10])
save_dict_lst(middle_file_conversation, conversations)

[{'role': 'user', 'content': 'Question: Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?\nContext: Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm). Trong khi chính phủ Pháp đang do dự về yêu cầu của Cao ủy Đông Dương Georges D\'Argenlieu về việc tăng quân và lập tức hành động quân sự chống lại người Việt, Valluy, người có chung quyết tâm với d\'Argenlieu về việc giữ sự hiện diện của Pháp tại Đông Dương, đã quyết định rằng cần phải 

# Gen answer

In [ ]:
conversation_df = pd.read_excel(middle_file_conversation)

conversation_queries = get_strs_in_df(conversation_df, "content")
conversation_combined_contexts = get_strs_in_df(conversation_df, "combined_context")

In [ ]:
# Call ChatGPT to generate answers
answers = [generate_answer(combined_context, query) for combined_context, query in zip(conversation_combined_contexts, conversation_queries)]

print(answers[:10])
save_dict_lst(middle_file_answer, answers, ["answer"])

ChatGPT:  Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam.
ChatGPT:  Trưởng phái đoàn Chính phủ
ChatGPT:  Trung Quốc, Liên Xô
ChatGPT:  rủi ro
ChatGPT:  biết hết các sách lược của Việt Nam và sử dụng chúng để ép Việt Nam ký hiệp định theo lợi ích của Trung Quốc.
ChatGPT:  khẳng định lại chủ quyền đối với 2 quần đảo này, nhắc lại lập trường của Việt Nam về việc giải quyết sự tranh chấp về 2 quần đảo giữa hai nước bằng thương lượng hoà bình.
ChatGPT:  hư hỏng quá, thoái hóa biến chất, chạy theo chức quyền, tiền và danh lợi, những người ấy đang làm cho một bộ phận không nhỏ trong nhân dân ta ngày càng giảm lòng tin vào Đảng ta
ChatGPT:  Gymnospermae
ChatGPT:  Không có thông tin nào trong bối cảnh về sự phát hiện của túi phôi trong thực vật có hoa.
ChatGPT:  thực vật hạt kín
ChatGPT:  trong kỷ Jura (146-200 Ma) và đa dạng hóa nhanh chóng trong kỷ Phấn Trắng (65-146 Ma)
ChatGPT:  thực vật có hoa
ChatGPT:  năm 1968 và công bố ở dạng đầy đủ năm 1981
ChatGPT:  không có thông tin

# Gen response

In [ ]:
retrieved_contexts_df = pd.read_excel(middle_file_retrieved_contexts)

retrieved_contexts_queries = get_strs_in_df(retrieved_contexts_df, 'user_query')
retrieved_contexts_contexts = get_str_lists_in_df(retrieved_contexts_df, 'retrieved_contexts')

answer_df = pd.read_excel(middle_file_answer)

answer_answers = get_strs_in_df(answer_df, "answer")

In [ ]:
reponses = [format_response(query, contexts, answer) for query, contexts, answer in zip(retrieved_contexts_queries, retrieved_contexts_contexts, answer_answers)]

print(reponses[:10])
save_dict_lst(middle_file_response, reponses)

[{'user_query': 'Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?', 'retrieved_contexts': ['Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', "Trong khi chính phủ Pháp đang do dự về yêu cầu của Cao ủy Đông Dương Georges D'Argenlieu về việc tăng quân và lập tức hành động quân sự chống lại người Việt, Valluy, người có chung quyết tâm với d'Argenlieu về việc giữ sự hiện diện của Pháp tại Đông Dương, đã quyết định rằng cần phải khiêu 

# Generate evaluations

In [ ]:
response_df = pd.read_excel(middle_file_response)

response_user_queries = get_strs_in_df(response_df, 'user_query')
response_retrieved_contexts = get_str_lists_in_df(response_df, 'retrieved_contexts')
response_responses = get_strs_in_df(response_df, 'response')

queries_df = pd.read_excel(input_file_queries)

response_references = get_strs_in_df(queries_df, 'answer')

In [ ]:
evaluation_inputs = [format_evaluation_input(user_query, retrieved_contexts, response, reference)  for user_query, retrieved_contexts, response, reference in zip(response_user_queries, response_retrieved_contexts, response_references, response_references)]

print(evaluation_inputs[:10])

# Convert to EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(evaluation_inputs)

[{'user_input': 'Phạm Văn Đồng đã đảm nhiệm vai trò gì trong chính phủ Việt Nam?', 'retrieved_contexts': ['Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', "Trong khi chính phủ Pháp đang do dự về yêu cầu của Cao ủy Đông Dương Georges D'Argenlieu về việc tăng quân và lập tức hành động quân sự chống lại người Việt, Valluy, người có chung quyết tâm với d'Argenlieu về việc giữ sự hiện diện của Pháp tại Đông Dương, đã quyết định rằng cần phải khiêu 

In [ ]:
# Define metrics for evaluation
metrics = [
    LLMContextRecall(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    SemanticSimilarity(embeddings=evaluator_embeddings)
]

In [ ]:
# Evaluate the dataset
evaluations = evaluate(dataset=evaluation_dataset, metrics=metrics)

# list of dictionaries to DataFrame
evaluations_df = evaluations.to_pandas()

# Save to Excel
evaluations_df.to_excel(output_file_evaluations, index=False)

Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[178]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-72T45A1iNIqqrUmE1gyhVMbh on tokens per min (TPM): Limit 200000, Used 199491, Requested 2447. Please try again in 581ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[317]: AttributeError('StringIO' object has no attribute 'decomposed_claims')
ERROR:ragas.executor:Exception raised in Job[346]: AttributeError('StringIO' object has no attribute 'statements')


### Analyze evaluations

In [ ]:
# Fill NaN with 0 before calculating the mean
evaluations_df_filled = evaluations_df.fillna(0)

# Analyze by means of context_recall, factual_correctness, faithfulness and semantic_similarity
evaluations_column_means_filled = evaluations_df_filled.mean(numeric_only=True)
print(evaluations_column_means_filled)

context_recall         0.760000
factual_correctness    0.957900
faithfulness           0.605833
semantic_similarity    0.999996
dtype: float64


### Analyze faithfulness

Faithfulness metric measures the factual consistency of the generated answer against the given context.

In [ ]:
create_summary(evaluations_df, True, summary_file_reordered_True)
create_summary(evaluations_df, False, summary_file_reordered_False)

Reordered summary exported to evaluation_summary_100_True_similar_dataset_without_context.xlsx
Reordered summary exported to evaluation_summary_100_False_similar_dataset_without_context.xlsx
